# 01 · Getting started: light up the wall from Python

> ⚠️ **Experimental.** This talks to the board using a reverse-engineered protocol.
> It works on the IICW Kilter board, but expect rough edges. See `docs/controlling-lights.md`.

**Before you start**
* `pip install -r requirements-lights.txt` (in the repo root)
* Your computer needs Bluetooth.
* The board only accepts **one** Bluetooth connection: close the Kilter app on everyone's phone.

In Jupyter you can use `await` directly in a cell; that's how all the Bluetooth calls work.

In [ ]:
# Make the repository root importable when the notebook runs from examples/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from boardlights import Board, Wall, scan

In [ ]:
# --- Configure your wall here ------------------------------------------------
# IICW default: Kilter Homewall (layout 8), 7x10 Mainline LED kit (size 18, 165 LEDs).
# Not sure? Run `python -m boardlights configs` to list all layout/size ids.
LAYOUT_ID = 8
SIZE_ID = 18
BOARD_ADDRESS = None   # e.g. "FC:0C:87:2B:53:25"; None = connect to the closest board

## 1. Find the board
This lists nearby devices that look like a climbing board, strongest signal first.

In [ ]:
for device in await scan(timeout=5):
    print(device)

Nothing found? Try `await scan(only_boards=False)` to see *every* Bluetooth device and look for the board's name.

## 2. Connect

In [ ]:
board = await Board.connect(address=BOARD_ADDRESS)
print("Connected to", board.address)

## 3. Light some LEDs
`board.show()` takes a dictionary `{led_position: colour}`.
Every LED that is **not** in the dictionary turns off.

Colours can be `"FF0000"`, `"#ff0000"` or `(255, 0, 0)`.

In [ ]:
await board.show({0: "FF0000"})   # LED 0 red

In [ ]:
await board.show({
    0: "FF0000",       # red
    1: (0, 255, 0),    # green
    2: "#0000ff",      # blue
})

### Colours are coarse
The board stores a colour in a single byte: 3 bits red, 3 bits green, 2 bits blue.
So there are only 8 × 8 × 4 = 256 colours, and blue has the fewest steps.

In [ ]:
from boardlights import encode_color
for c in ["FF0000", "800000", "400000", "200000", "0000FF", "000080"]:
    print(c, "->", format(encode_color(c), "08b"))

## 4. Light every LED, one colour

In [ ]:
wall = Wall(LAYOUT_ID, SIZE_ID)
print(len(wall.positions), "LEDs on this wall")
await board.show({p: "FFFFFF" for p in wall.positions})

## 5. Turn everything off and disconnect
Always disconnect when you're done, otherwise nobody else (or the app) can connect.

In [ ]:
await board.clear()
await board.disconnect()

## Tip: `async with`
This connects and **always** disconnects, even if your code crashes halfway:

In [ ]:
async with await Board.connect(address=BOARD_ADDRESS) as board:
    await board.show({10: "FFFF00"})

## Under the hood
Curious what is actually sent? `build_frame` turns the dictionary into bytes:
`[0x01, length, checksum, 0x02, packet_type, pos_low, pos_high, colour, ..., 0x03]`.

In [ ]:
from boardlights import build_frame
print(list(build_frame({5: "800080"})))

Next: **02_climbs_and_holds.ipynb**: use real hold coordinates and light climbs from the database.